### Import Packages

In [1]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import re, os
import cv2

### Load Data

In [2]:
# Load the original metadata.csv
csv_path = "mimic-cxr-dataset/metadata.csv"
df_metadata = pd.read_csv(csv_path)

In [3]:
# Create new dataframe for image and report paths
def get_image_path(row):
    return f"mimic-cxr-dataset/official_data_iccv_final/files/p{str(row.subject_id)[:2]}/p{row.subject_id}/s{row.study_id}/{row.dicom_id}.jpg"

def get_report_path(row):
    return f"mimic-cxr-dataset/mimic-cxr-reports/files/p{str(row.subject_id)[:2]}/p{row.subject_id}/s{row.study_id}.txt"

In [4]:
# Create a new dataframe
df_paths = df_metadata.copy()
df_paths["image_path"] = df_paths.apply(get_image_path, axis=1)
df_paths["report_path"] = df_paths.apply(get_report_path, axis=1)

In [5]:
# Group by study_id to collect image paths
df_grouped = df_paths.groupby("study_id").agg({
    "subject_id": "first",
    "report_path": "first",
    "image_path": list,
    "ViewPosition": list
}).reset_index()

df_grouped['image_path'] = df_grouped['image_path'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)


In [6]:
# View dataframe
df_grouped.head()

,study_id,subject_id,report_path,image_path,ViewPosition
0,50000198,16548129,mimic-cxr-dataset/mimic-cxr-reports/files/p16/...,mimic-cxr-dataset/official_data_iccv_final/fil...,[AP]
1,50000801,18780736,mimic-cxr-dataset/mimic-cxr-reports/files/p18/...,mimic-cxr-dataset/official_data_iccv_final/fil...,"[LL, PA, PA]"
2,50001143,11266603,mimic-cxr-dataset/mimic-cxr-reports/files/p11/...,mimic-cxr-dataset/official_data_iccv_final/fil...,"[LL, PA]"
3,50001598,16861367,mimic-cxr-dataset/mimic-cxr-reports/files/p16/...,mimic-cxr-dataset/official_data_iccv_final/fil...,[AP]
4,50001683,19977310,mimic-cxr-dataset/mimic-cxr-reports/files/p19/...,mimic-cxr-dataset/official_data_iccv_final/fil...,"[PA, LATERAL]"


### Extracting text from reports

In [7]:
# function to extract caption of findings
def extract_findings(report_path):
    try:
        with open(report_path, "r") as file:
            text = file.read()

        match = re.search(r"FINDINGS:\s*(.*?)\s*IMPRESSION:", text, re. DOTALL| re.IGNORECASE)

        if not match:
            match = re.search(r'FINDINGS:\s*(.*)', text, re.DOTALL | re.IGNORECASE)

        if match:
            findings_text = match.group(1)
            findings_text = findings_text.replace('\n', '')
            findings_text = findings_text.replace('_', '')
            findings_text = re.sub(r'\s+', ' ', findings_text)

            return findings_text.strip()
        
        return None
    
    except Exception:
        return None

In [8]:
# Apply captions to the dataset
df_grouped['findings_text'] = df_grouped['report_path'].apply(extract_findings)

In [9]:
# function to extract caption of impressions
def extract_impression(report_path):
    try:
        with open(report_path, "r") as file:
            text = file.read()

        match = re.search(r"IMPRESSION:\s*(.*)", text, re. DOTALL| re.IGNORECASE)

        if match:
            findings_text = match.group(1).strip()
            findings_text = findings_text.replace('\n', '')
            findings_text = findings_text.replace('_', '')
            findings_text = re.sub(r'\s+', ' ', findings_text)

            return findings_text
        
        return None
    
    except Exception:
        return None

In [10]:
# Apply impressions to the dataset
df_grouped['impressions_text'] = df_grouped['report_path'].apply(extract_impression)

In [11]:
# joins the texts into one
df_grouped['all_texts'] = df_grouped.apply(
    lambda row: ' '.join([str(text) for text in [row['findings_text'], row['impressions_text']] if pd.notnull(text)]),
    axis=1
)

In [12]:
# View dataframe
df_grouped.head()

,study_id,subject_id,report_path,image_path,ViewPosition,findings_text,impressions_text,all_texts
0,50000198,16548129,mimic-cxr-dataset/mimic-cxr-reports/files/p16/...,mimic-cxr-dataset/official_data_iccv_final/fil...,[AP],Heart size is normal. The mediastinal and hila...,No acute cardiopulmonary abnormality.,Heart size is normal. The mediastinal and hila...
1,50000801,18780736,mimic-cxr-dataset/mimic-cxr-reports/files/p18/...,mimic-cxr-dataset/official_data_iccv_final/fil...,"[LL, PA, PA]",None,In comparison with the earlier study of this d...,In comparison with the earlier study of this d...
2,50001143,11266603,mimic-cxr-dataset/mimic-cxr-reports/files/p11/...,mimic-cxr-dataset/official_data_iccv_final/fil...,"[LL, PA]",None,"As compared to the previous image, there is vi...","As compared to the previous image, there is vi..."
3,50001598,16861367,mimic-cxr-dataset/mimic-cxr-reports/files/p16/...,mimic-cxr-dataset/official_data_iccv_final/fil...,[AP],"In comparison with the study of , the endotrac...",None,"In comparison with the study of , the endotrac..."
4,50001683,19977310,mimic-cxr-dataset/mimic-cxr-reports/files/p19/...,mimic-cxr-dataset/official_data_iccv_final/fil...,"[PA, LATERAL]",PA and lateral views of the chest were provide...,No acute findings in the chest.,PA and lateral views of the chest were provide...


### Dataset cleaning

In [13]:
# lines in which there is no intuitive pattern to extract the text
count_both_none = df_grouped[df_grouped[['findings_text', 'impressions_text']].isnull().all(axis=1)].shape[0]
print("Lines with reports is unidentifiable:", count_both_none)

print(f"DataFrame Before: {len(df_grouped)} lines")
df_grouped = df_grouped[~df_grouped[['findings_text', 'impressions_text']].isnull().all(axis=1)]
print(f"DataFrame After: {len(df_grouped)} lines")

Lines with reports is unidentifiable: 2587
DataFrame Before: 52434 lines
DataFrame After: 49847 lines


In [14]:
# Check if the image is readable
def check_image_validity(path):
    if not isinstance(path, str) or not os.path.isfile(path):
        return False
    img = cv2.imread(path)
    return img is not None

In [15]:
df_grouped['image_valid'] = df_grouped['image_path'].apply(check_image_validity)

num_invalid = (~df_grouped['image_valid']).sum()
print(f"Invalid or missing images: {num_invalid}")


Invalid or missing images: 0


In [16]:
# View dataframe
df_grouped.head()

,study_id,subject_id,report_path,image_path,ViewPosition,findings_text,impressions_text,all_texts,image_valid
0,50000198,16548129,mimic-cxr-dataset/mimic-cxr-reports/files/p16/...,mimic-cxr-dataset/official_data_iccv_final/fil...,[AP],Heart size is normal. The mediastinal and hila...,No acute cardiopulmonary abnormality.,Heart size is normal. The mediastinal and hila...,True
1,50000801,18780736,mimic-cxr-dataset/mimic-cxr-reports/files/p18/...,mimic-cxr-dataset/official_data_iccv_final/fil...,"[LL, PA, PA]",None,In comparison with the earlier study of this d...,In comparison with the earlier study of this d...,True
2,50001143,11266603,mimic-cxr-dataset/mimic-cxr-reports/files/p11/...,mimic-cxr-dataset/official_data_iccv_final/fil...,"[LL, PA]",None,"As compared to the previous image, there is vi...","As compared to the previous image, there is vi...",True
3,50001598,16861367,mimic-cxr-dataset/mimic-cxr-reports/files/p16/...,mimic-cxr-dataset/official_data_iccv_final/fil...,[AP],"In comparison with the study of , the endotrac...",None,"In comparison with the study of , the endotrac...",True
4,50001683,19977310,mimic-cxr-dataset/mimic-cxr-reports/files/p19/...,mimic-cxr-dataset/official_data_iccv_final/fil...,"[PA, LATERAL]",PA and lateral views of the chest were provide...,No acute findings in the chest.,PA and lateral views of the chest were provide...,True


### Data division into training and testing (80%/20%)

In [17]:
# Ensures that studies of the same patient do not mix
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(df_grouped, groups=df_grouped["subject_id"]))

df_train = df_grouped.iloc[train_idx]
df_test = df_grouped.iloc[test_idx]

In [18]:
# View splits
print(f"Train Data: {len(df_train)} lines")
print(f"Test Data: {len(df_test)} lines")

Train Data: 39961 lines
Test Data: 9886 lines


In [19]:
# Creates a dataset to store the data
results_path = "resources"

if not os.path.exists(results_path):
    os.makedirs(results_path)

# Save dataframes in csv files
df_grouped.to_csv(f"{results_path}/all_data.csv", index=False)
df_train.to_csv(f"{results_path}/train_data.csv", index=False)
df_test.to_csv(f"{results_path}/test_data.csv", index=False)